# Question 2 — Brain MRI: N4 Bias Correction + K-means Segmentation
Using ANTsPy to apply N4 bias field correction and K-means tissue segmentation on a T2 brain MRI.

> **Note:** ANTsPy is a large package (~500 MB). Installation may take several minutes.

In [ ]:
# Step 0: Install ANTsPy (run once)
import subprocess
subprocess.run(["pip", "install", "antspyx"], check=True)

In [ ]:
# Step 1: Imports
import ants
import numpy as np

In [ ]:
# Step 2: Load brain image
brain = ants.image_read("brain.nii.gz")   # change path if needed
print(f"Image shape:   {brain.shape}")
print(f"Image spacing: {brain.spacing}")

In [ ]:
# Step 3: Create brain mask
# Needed for accurate bias correction — excludes air and skull
brain_mask = ants.get_mask(brain)
print("Brain mask created.")

In [ ]:
# Step 4: N4 Bias Field Correction
# Removes low-frequency intensity inhomogeneity common in 3T MRI
print("Running N4 bias field correction...")
brain_n4 = ants.n4_bias_field_correction(
    brain,
    mask=brain_mask,
    convergence={"iters": [50, 50, 50, 50], "tol": 1e-7},
    shrink_factor=4,
    return_bias_field=False,
    verbose=True
)
print("N4 correction complete.")

In [ ]:
# Step 5: K-means Segmentation
# k=3 → CSF (1), Gray Matter (2), White Matter (3)
print("Running k-means segmentation (k=3)...")
seg = ants.kmeans_segmentation(
    image=brain_n4,
    k=3,       # 3 tissue classes standard for T2 brain MRI
    kmask=brain_mask,
    mrf=0.1    # light spatial regularization to smooth mislabeled voxels
)
segmented_img = seg["segmentation"]
print("Segmentation complete.")

In [ ]:
# Step 6: Print label summary
label_vals, counts = np.unique(segmented_img.numpy().flatten().astype(int), return_counts=True)
label_names = {0: "Background", 1: "CSF", 2: "Gray Matter", 3: "White Matter"}

print("=" * 40)
print("  SEGMENTATION SUMMARY")
print("=" * 40)
for u, c in zip(label_vals, counts):
    name = label_names.get(u, f"Class {u}")
    print(f"  Label {u} ({name:>12s}): {c:,} voxels")
print("=" * 40)

In [ ]:
# Step 7: Save segmented NIFTI file
ants.image_write(segmented_img, "brain_kmeans_segmented.nii.gz")
print("Segmented file saved: brain_kmeans_segmented.nii.gz")